# Smartphone Addiction Prediction — Kaggle Competition

## Objective

Build a production-ready machine learning pipeline that achieves a competitive Kaggle leaderboard score and can be deployed as a real-world prediction service.

### Goals

* Maximize leaderboard performance
* Build a reusable preprocessing pipeline
* Compare multiple classification models
* Optimize hyperparameters
* Create a deployable inference pipeline
* Serve predictions through a FastAPI API
* Containerize with Docker
* Publish a portfolio-ready GitHub repository


In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

In [2]:
DATA_DIR = Path("../data")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Submission:", sample_submission.shape)

Train: (691369, 14)
Test: (296302, 13)
Submission: (296302, 2)


In [3]:
TARGET = "addicted_label"
ID_COL = "id"

X = train_df.drop(columns=[TARGET, ID_COL]).copy()
y = train_df[TARGET].copy()

X_test = test_df.drop(columns=[ID_COL]).copy()

categorical_features = X.select_dtypes(include="object").columns.tolist()
numerical_features = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical:", categorical_features)
print("Numerical:", numerical_features)

Categorical: ['gender', 'stress_level', 'academic_work_impact']
Numerical: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time']


In [4]:
for col in categorical_features:
    X[col] = X[col].fillna("Missing").astype(str)
    X_test[col] = X_test[col].fillna("Missing").astype(str)

In [5]:
train_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,691369.0,NaN,NaN,NaN,345684.0,199581.183466,0.0,172842.0,345684.0,518526.0,691368.0
age,662440.0,NaN,NaN,NaN,26.615408,5.153162,18.0,22.0,27.0,31.0,35.0
daily_screen_time_hours,595515.0,NaN,NaN,NaN,7.640865,2.721446,0.5,5.48,7.77,9.84,15.0
social_media_hours,557374.0,NaN,NaN,NaN,2.471038,1.316137,0.0,1.45,2.31,3.37,8.0
gaming_hours,564548.0,NaN,NaN,NaN,1.459265,0.934552,0.0,0.7,1.33,2.09,4.0
work_study_hours,639851.0,NaN,NaN,NaN,2.366971,1.258797,0.0,1.36,2.2,3.2,6.0
sleep_hours,646889.0,NaN,NaN,NaN,6.804334,1.234512,4.5,5.78,6.8,7.87,9.0
notifications_per_day,623785.0,NaN,NaN,NaN,145.8949,65.917556,20.0,93.0,150.0,204.0,250.0
app_opens_per_day,610659.0,NaN,NaN,NaN,102.636781,48.09397,15.0,64.0,104.0,145.0,180.0
weekend_screen_time,579306.0,NaN,NaN,NaN,9.479866,2.856006,0.51,7.28,9.58,11.75,17.56


In [6]:
cat_features = [X.columns.get_loc(col) for col in categorical_features]

print(cat_features)

[9, 10, 11]


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

auc_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features
    )

    valid_pred = model.predict_proba(X_valid)[:, 1]

    auc = roc_auc_score(y_valid, valid_pred)
    auc_scores.append(auc)

    print(f"Fold {fold}: {auc:.6f}")

print("\nMean CV AUC:", np.mean(auc_scores))

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

# Categorical column indices for CatBoost
cat_features = [X.columns.get_loc(col) for col in categorical_features]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_scores = []

for train_idx, valid_idx in cv.split(X, y):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features
    )

    preds = model.predict_proba(X_valid)[:, 1]
    auc_scores.append(roc_auc_score(y_valid, preds))

print("Fold AUC:", auc_scores)
print("Mean AUC:", np.mean(auc_scores))

In [ ]:
final_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200
)

final_model.fit(X, y, cat_features=cat_features)

test_pred = final_model.predict_proba(X_test)[:, 1]

submission = sample_submission.copy()
submission[sample_submission.columns[1]] = test_pred

submission.to_csv("../submissions/catboost_v1.csv", index=False)

print(submission.head())